# Exercise 64 — SQLite with `sqlite3`

## Concept

SQLite is a file-based database — no server. The `sqlite3` module is in the standard library, so it's the easiest way to play with real SQL.

```python
import sqlite3

conn = sqlite3.connect("app.db")            # or ":memory:" for in-memory
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT
    )
""")

# ⚠ ALWAYS use parameterized queries — never f-strings or % formatting
cur.execute("INSERT INTO users (name, email) VALUES (?, ?)", ("Alice", "a@x.com"))
conn.commit()

cur.execute("SELECT id, name, email FROM users WHERE name = ?", ("Alice",))
row  = cur.fetchone()           # single row tuple, or None
rows = cur.fetchall()           # list of tuples
for row in cur:                 # iterate (streams from DB)
    ...

conn.close()
```

### Parameterized queries — non-negotiable

**Never** do this:

```python
cur.execute(f"SELECT * FROM users WHERE name = '{name}'")   # SQL INJECTION
```

If `name = "' OR 1=1 --"`, you've leaked the whole table. Always use `?` placeholders, pass values as a tuple. The driver escapes them safely.

### `with conn:` for transactions

```python
with conn:
    cur.execute("INSERT INTO ...")
    cur.execute("INSERT INTO ...")
# block exits cleanly → commit. Exception → rollback.
```

Note: this commits the **transaction**, not closes the connection. Still call `conn.close()` when done.

## Task 1 — Create, insert, select

Write three functions against an **in-memory** SQLite database (`":memory:"`):

```python
def setup_db() -> sqlite3.Connection:
    """Open an in-memory DB and create a 'users' table:
       id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT.
       Return the connection."""

def insert_user(conn, name: str, email: str | None) -> int:
    """Insert one user. Return the auto-generated id."""

def get_user(conn, user_id: int) -> tuple | None:
    """Return (id, name, email) or None if not found."""
```

Use parameterized SQL throughout. `cur.lastrowid` gives the inserted id.

In [ ]:
import sqlite3

def setup_db() -> sqlite3.Connection:
    pass  # TODO

def insert_user(conn, name: str, email: str | None) -> int:
    pass  # TODO

def get_user(conn, user_id: int) -> tuple | None:
    pass  # TODO

conn = setup_db()
id1 = insert_user(conn, "Alice", "a@x.com")
id2 = insert_user(conn, "Bob", None)
assert id1 == 1 and id2 == 2
assert get_user(conn, 1) == (1, "Alice", "a@x.com")
assert get_user(conn, 2) == (2, "Bob", None)
assert get_user(conn, 999) is None
conn.close()
print("ok")


## Task 2 — Bulk insert with `executemany`

`cursor.execute()` does one row at a time. `cursor.executemany()` takes a sequence of parameter tuples — much faster than a Python `for` loop calling `execute`.

```python
def insert_many_users(conn, users: list[tuple[str, str | None]]) -> int:
    """Insert all users via executemany. Return the number of rows inserted."""
```

Don't forget to commit (or use `with conn:`).

Verify with a `SELECT COUNT(*)` afterwards.

In [ ]:
import sqlite3

def insert_many_users(conn, users: list[tuple[str, str | None]]) -> int:
    pass  # TODO

conn = setup_db()
n = insert_many_users(conn, [("A", "a@x"), ("B", None), ("C", "c@x")])
assert n == 3
count = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
assert count == 3
conn.close()
print("ok")


## Task 3 — Parameterized SELECT (NOT f-string!)

Write a function that finds users by partial name match using SQL `LIKE`.

```python
def search_users(conn, name_pattern: str) -> list[tuple]:
    """Return (id, name, email) for users whose name matches the SQL LIKE pattern.
    The pattern is passed as-is — caller provides any % wildcards.
    Use parameter binding — do NOT format the pattern into the SQL string."""
```

Expected behavior:
- `search_users(conn, "%lice%")` returns rows whose name contains `"lice"` (e.g., Alice)
- A pattern containing `'` or `--` does not break or inject — the driver escapes it

In [ ]:
import sqlite3

def search_users(conn, name_pattern: str) -> list[tuple]:
    pass  # TODO

conn = setup_db()
insert_many_users(conn, [("Alice", "a@x"), ("Bob", None), ("Carol", "c@x"), ("Alicia", "al@x")])

matches = search_users(conn, "%lic%")    # Alice + Alicia
names = [row[1] for row in matches]
assert set(names) == {"Alice", "Alicia"}

# An injection attempt must be treated as a literal string, not interpreted
weird = search_users(conn, "' OR 1=1 --")
assert weird == []   # the literal pattern doesn't match any name
conn.close()
print("ok")


## Task 4 — Transaction rollback

Write a function that inserts a batch of users **atomically**: if any row fails (e.g., `name` is `None`, which violates the `NOT NULL` constraint), the entire batch must be rolled back.

```python
def atomic_insert(conn, users: list[tuple[str | None, str | None]]) -> bool:
    """Try to insert all users in a single transaction.
    Return True if everything succeeded, False if any row caused a rollback.
    After False, the users table must contain no rows from this batch.
    """
```

Hint: use a `try` block, `conn.commit()` on success, `conn.rollback()` in `except`. Or use `with conn:` which handles commit/rollback automatically — but in that case the exception propagates, and you'd wrap the `with` in `try/except`.

Catch only `sqlite3.IntegrityError` (the constraint-violation type).

In [ ]:
import sqlite3

def atomic_insert(conn, users: list[tuple[str | None, str | None]]) -> bool:
    pass  # TODO

# Happy path
conn = setup_db()
ok = atomic_insert(conn, [("Alice", "a@x"), ("Bob", None)])
assert ok is True
assert conn.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 2
conn.close()

# Failure path: second row violates NOT NULL on name → whole batch rolled back
conn = setup_db()
ok = atomic_insert(conn, [("Alice", "a@x"), (None, "b@x")])
assert ok is False
assert conn.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0   # no partial insert
conn.close()
print("ok")
